In [1]:
import pandas as pd

skills_df = pd.read_csv("dataset/skills.csv")
courses_df = pd.read_csv("dataset/Online_Courses.csv")

# Create description column from Short Intro
courses_df["description"] = courses_df["Short Intro"].fillna("")

# Clean Skills column
courses_df["Skills"] = courses_df["Skills"].fillna("").str.lower()
courses_df["Skills"] = courses_df["Skills"].apply(
    lambda x: [s.strip() for s in x.split(",")] if x else []
)

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Skill embeddings
skill_embeddings = model.encode(skills_df["description"].tolist())

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(skill_embeddings)

edges = []
threshold = 0.6

for i in range(len(skills_df)):
    for j in range(i+1, len(skills_df)):
        if sim_matrix[i][j] > threshold:
            edges.append((
                skills_df.iloc[i]["skill"],
                skills_df.iloc[j]["skill"]
            ))

In [4]:
import networkx as nx

G = nx.Graph()
G.add_edges_from(edges)

print("Total Nodes:", G.number_of_nodes())
print("Total Edges:", G.number_of_edges())

Total Nodes: 359
Total Edges: 4971


In [5]:
known_skills = ["python"]
goal = "deep learning"

In [6]:
user_vec = model.encode([" ".join(known_skills)])[0]

scores = cosine_similarity([user_vec], skill_embeddings)[0]

skill_scores = list(zip(skills_df["skill"], scores))

# Sort by relevance
skill_scores = sorted(skill_scores, key=lambda x: x[1], reverse=True)

# Remove known skills
recommended_skills = [
    skill for skill, score in skill_scores
    if skill not in known_skills
]

In [7]:
try:
    path = nx.shortest_path(G, source=known_skills[0], target=goal)
except:
    path = ["No path found"]

In [8]:
print("\n🔍 Explanation:")
if path != ["No path found"]:
    print(" → ".join(path))
else:
    print("No connection found in graph")


🔍 Explanation:
No connection found in graph


In [9]:
def recommend_courses(skill):
    return courses_df[
        courses_df["Skills"].apply(lambda x: skill in x)
    ][["Title"]].head(3)

print("\n📚 Recommended Courses:")

if path != ["No path found"]:
    for skill in path:
        print(f"\nSkill: {skill}")
        print(recommend_courses(skill))


📚 Recommended Courses:


In [10]:
print("\n🎯 Top Recommended Skills:")
print(recommended_skills[:5])


🎯 Top Recommended Skills:
['python programming', 'python libraries', 'python tools', 'python scripting', 'python programming skills']
